In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import numpy as np
import pandas as pd

Path to ROCo dataset files

In [3]:
train_images_path = "/content/drive/MyDrive/ROCo/radiology/img"
train_captions_path = "/content/drive/MyDrive/ROCo/radiology/traindata.csv"
val_images_path = "/content/drive/MyDrive/ROCo/validation/radiology/img"
val_captions_path = "/content/drive/MyDrive/ROCo/validation/radiology/valdata.csv"

Loading and Preprocessing of data

In [4]:
def load_captions(captions_path):
    df = pd.read_csv(captions_path)
    return df

In [5]:
from keras.applications import VGG19
from keras.applications.vgg19 import preprocess_input
from keras.preprocessing.image import load_img, img_to_array
from keras.models import Model

In [6]:
def preprocess_images(images_path):
    vgg19 = VGG19(weights="imagenet")
    feature_extractor = Model(inputs=vgg19.input, outputs=vgg19.get_layer("fc2").output)

    features = {}
    for img_name in os.listdir(images_path):
        img_path = os.path.join(images_path, img_name)
        img = load_img(img_path, target_size=(224, 224))
        img_array = img_to_array(img)
        img_array = np.expand_dims(img_array, axis=0)
        img_array = preprocess_input(img_array)

        feature = feature_extractor.predict(img_array)
        features[img_name] = feature
    return features

Feature extraction for images

In [7]:
train_features = preprocess_images(train_images_path)
val_features = preprocess_images(val_images_path)

574710816/574710816 ━━━━━━━━━━━━━━━━━━━━ 15s 0us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 741ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 754ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 755ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 799ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 765ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 789ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 752ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 753ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 917ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 762ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 726ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 751ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 752ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 762ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 747ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 781ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 754ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 749

Loading Captions

In [8]:
train_captions = load_captions(train_captions_path)
val_captions = load_captions(val_captions_path)

In [9]:
from tensorflow.keras.preprocessing.text import Tokenizer
from keras.utils import pad_sequences
from sklearn.model_selection import train_test_split
tokenizer = Tokenizer()
tokenizer.fit_on_texts(train_captions['caption'])

In [10]:
vocab_size = len(tokenizer.word_index) + 1
max_len = max([len(c.split()) for c in train_captions['caption']])

In [11]:
def create_sequences(features, captions, tokenizer, max_len):
    X1, X2, y = [], [], []
    missing_keys = []
    valid_keys = set(features.keys())

    for label, caption in zip(captions['name'], captions['caption']):
        if label in valid_keys:
            seq = tokenizer.texts_to_sequences([caption])[0]
            if len(seq) < 2:
                print(f"Skipping caption '{caption}' for label '{label}' as it is too short.")
                continue

            for i in range(1, len(seq)):
                in_seq, out_seq = seq[:i], seq[i]
                in_seq = pad_sequences([in_seq], maxlen=max_len)[0]
                out_seq_one_hot = np.zeros(vocab_size)
                out_seq_one_hot[out_seq] = 1

                feature = features[label].flatten() if len(features[label].shape) > 1 else features[label]
                X1.append(feature)
                X2.append(in_seq)
                y.append(out_seq_one_hot)
        else:
            missing_keys.append(label)

    if missing_keys:
        print(f"Number of missing keys: {len(missing_keys)}")
        print(f"Missing keys: {missing_keys[:10]}{'...' if len(missing_keys) > 10 else ''}")  # Print first 10 missing keys

    return np.array(X1), np.array(X2), np.array(y)


In [12]:
X1_train, X2_train, y_train = create_sequences(train_features, train_captions, tokenizer, max_len)
print(f"X1_train shape: {X1_train.shape}")
print(f"X2_train shape: {X2_train.shape}")
print(f"y_train shape: {y_train.shape}")


Number of missing keys: 65407
Missing keys: ['PMC4083729_AMHSR-4-14-g002.jpg', 'PMC2837471_IJD2009-150251.001.jpg', 'PMC2505281_11999_2007_30_Fig6_HTML.jpg', 'PMC3745845_IJD2013-683423.005.jpg', 'PMC4917066_amjcaserep-17-301-g001.jpg', 'PMC4805615_13244_2016_481_Fig12_HTML.jpg', 'PMC2584650_1757-1626-1-193-1.jpg', 'PMC3283944_JISP-15-414-g006.jpg', 'PMC4946383_HI-10-1-25-g003.jpg', 'PMC5646151_TOORTHJ-11-882_F2.jpg']...
X1_train shape: (1294, 4096)
X2_train shape: (1294, 410)
y_train shape: (1294, 37660)


In [13]:
print(f"Number of valid samples: {len(X1_train)}")

Number of valid samples: 1294


In [14]:
X1_train, X2_train, y_train = create_sequences(train_features, train_captions, tokenizer, max_len)
X1_val, X2_val, y_val = create_sequences(val_features, val_captions, tokenizer, max_len)

Number of missing keys: 65407
Missing keys: ['PMC4083729_AMHSR-4-14-g002.jpg', 'PMC2837471_IJD2009-150251.001.jpg', 'PMC2505281_11999_2007_30_Fig6_HTML.jpg', 'PMC3745845_IJD2013-683423.005.jpg', 'PMC4917066_amjcaserep-17-301-g001.jpg', 'PMC4805615_13244_2016_481_Fig12_HTML.jpg', 'PMC2584650_1757-1626-1-193-1.jpg', 'PMC3283944_JISP-15-414-g006.jpg', 'PMC4946383_HI-10-1-25-g003.jpg', 'PMC5646151_TOORTHJ-11-882_F2.jpg']...
Number of missing keys: 8130
Missing keys: ['PMC3970251_CRIONM2014-931546.003.jpg', 'PMC2766744_cios-1-176-g005.jpg', 'PMC3789931_poljradiol-78-3-35-g001.jpg', 'PMC2676075_p147_fig4a.jpg', 'PMC5292123_CRIGM2017-1710501.002.jpg', 'PMC4756892_CMJ-128-2946-g004.jpg', 'PMC2494540_1757-1626-1-52-1.jpg', 'PMC3339065_NAJMS-2-392-g001.jpg', 'PMC2636159_IndianJOphthalmol-56-269-g003.jpg', 'PMC5625558_1657-9534-cm-48-02-00088-gf1.jpg']...


In [15]:
from keras.layers import Input, LSTM, Embedding, Dense, Dropout, RNN, SimpleRNN
from keras.models import Model
def build_lstm_model(vocab_size, max_len):
    inputs1 = Input(shape=(4096,))
    fe1 = Dropout(0.5)(inputs1)
    fe2 = Dense(256, activation='relu')(fe1)

    inputs2 = Input(shape=(max_len,))
    se1 = Embedding(vocab_size, 256, mask_zero=True)(inputs2)
    se2 = LSTM(256)(se1)

    decoder1 = Dense(256, activation='relu')((fe2))
    outputs = Dense(vocab_size, activation='softmax')(decoder1)
    model = Model(inputs=[inputs1, inputs2], outputs=outputs)
    model.compile(loss='categorical_crossentropy', optimizer='adam')
    return model

In [16]:
def build_rnn_model(vocab_size, max_len):
    inputs1 = Input(shape=(4096,))
    fe1 = Dropout(0.5)(inputs1)
    fe2 = Dense(256, activation='relu')(fe1)

    inputs2 = Input(shape=(max_len,))
    se1 = Embedding(vocab_size, 256, mask_zero=True)(inputs2)
    se2 = SimpleRNN(256)(se1)

    decoder1 = Dense(256, activation='relu')((fe2))
    outputs = Dense(vocab_size, activation='softmax')(decoder1)
    model = Model(inputs=[inputs1, inputs2], outputs=outputs)
    model.compile(loss='categorical_crossentropy', optimizer='adam')
    return model

In [17]:
lstm_model = build_lstm_model(vocab_size, max_len)
lstm_model.fit([X1_train, X2_train], y_train, epochs=20, batch_size=64, validation_data=([X1_val, X2_val], y_val))

Epoch 1/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 10s 374ms/step - loss: 9.3022 - val_loss: 8.4321
Epoch 2/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 6s 272ms/step - loss: 5.7695 - val_loss: 8.6546
Epoch 3/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 8s 383ms/step - loss: 4.9139 - val_loss: 8.5669
Epoch 4/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 9s 325ms/step - loss: 4.2184 - val_loss: 8.6953
Epoch 5/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 8s 379ms/step - loss: 3.8125 - val_loss: 8.5513
Epoch 6/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 8s 271ms/step - loss: 3.6984 - val_loss: 8.7472
Epoch 7/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 10s 269ms/step - loss: 3.6384 - val_loss: 8.5132
Epoch 8/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 12s 354ms/step - loss: 3.5166 - val_loss: 8.4626
Epoch 9/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 10s 345ms/step - loss: 3.5339 - val_loss: 8.4941
Epoch 10/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 9s 268ms/step - loss: 3.4551 - val_loss: 8.6353
Epoch 11/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 8s 368ms/step - loss: 3.4329 - val_loss: 8.3855
Epoch 12/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 9s 316ms/

In [18]:
rnn_model = build_rnn_model(vocab_size, max_len)
rnn_model.fit([X1_train, X2_train], y_train, epochs=20, batch_size=64, validation_data=([X1_val, X2_val], y_val))

Epoch 1/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 10s 380ms/step - loss: 9.2854 - val_loss: 8.3376
Epoch 2/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 10s 381ms/step - loss: 5.7342 - val_loss: 8.6799
Epoch 3/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 9s 308ms/step - loss: 4.8550 - val_loss: 8.7204
Epoch 4/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 11s 340ms/step - loss: 4.2059 - val_loss: 8.5802
Epoch 5/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 12s 426ms/step - loss: 3.7847 - val_loss: 8.6912
Epoch 6/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 8s 296ms/step - loss: 3.6925 - val_loss: 8.4807
Epoch 7/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 9s 425ms/step - loss: 3.6267 - val_loss: 8.8520
Epoch 8/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 8s 309ms/step - loss: 3.5531 - val_loss: 8.6291
Epoch 9/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 10s 320ms/step - loss: 3.5261 - val_loss: 8.5212
Epoch 10/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 12s 408ms/step - loss: 3.4733 - val_loss: 8.4788
Epoch 11/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 9s 363ms/step - loss: 3.4388 - val_loss: 8.4851
Epoch 12/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 9s 315m

In [19]:
lstm_loss = lstm_model.evaluate([X1_val, X2_val], y_val)
rnn_loss = rnn_model.evaluate([X1_val, X2_val], y_val)

31/31 ━━━━━━━━━━━━━━━━━━━━ 2s 57ms/step - loss: 8.4513
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 35ms/step - loss: 8.4488


In [20]:
print(f"LSTM Loss: {lstm_loss}")
print(f"RNN Loss: {rnn_loss}")

LSTM Loss: 8.439229965209961
RNN Loss: 8.46412181854248


Evaluation Metrics

In [26]:
y_pred = lstm_model.predict([X1_val, X2_val])
y_pred_classes = np.argmax(y_pred, axis=-1)
y_val_classes = np.argmax(y_val, axis=-1)

31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step


In [27]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

In [28]:
accuracy = accuracy_score(y_val_classes, y_pred_classes)
precision = precision_score(y_val_classes, y_pred_classes, average='weighted')
recall = recall_score(y_val_classes, y_pred_classes, average='weighted')
f1 = f1_score(y_val_classes, y_pred_classes, average='weighted')
roc_auc = roc_auc_score(y_val, y_pred, multi_class='ovr')

/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_ranking.py:375: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_ranking.py:375: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  w

In [29]:
print("Accuracy: ", accuracy)
print("Precision: ", precision)
print("Recall: ", recall)
print("F1-Score: ", f1)
print("AUC-ROC: ", roc_auc)

Accuracy:  0.05402650356778797
Precision:  0.006297448905337785
Recall:  0.05402650356778797
F1-Score:  0.010921632449059757
AUC-ROC:  nan


In [30]:
from sklearn.metrics import classification_report
unique_classes = np.unique(y_val_classes)
target_names = [str(cls) for cls in unique_classes]
report = classification_report(y_val_classes, y_pred_classes, target_names=target_names, labels=unique_classes)
print("Classification Report: ")
print(report)

Classification Report: 
              precision    recall  f1-score   support

           1       0.07      0.72      0.12        64
           2       0.03      0.07      0.04        30
           3       0.00      0.00      0.00        25
           4       0.00      0.00      0.00        22
           5       0.00      0.00      0.00        26
           6       0.00      0.00      0.00        14
           7       0.00      0.00      0.00         9
           8       0.00      0.00      0.00        13
           9       0.00      0.00      0.00        13
          10       0.00      0.00      0.00        14
          11       0.03      0.10      0.05        10
          12       0.00      0.00      0.00        14
          13       0.00      0.00      0.00        10
          14       0.00      0.00      0.00         9
          15       0.06      0.17      0.09         6
          16       0.00      0.00      0.00        17
          17       0.00      0.00      0.00         4
   

/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [31]:
!pip install rouge_score

In [34]:
from rouge_score import rouge_scorer

def decode_sequence(input_seq):
    return str(np.argmax(input_seq))

decoded_preds = [decode_sequence(seq) for seq in y_pred]
decoded_vals = [decode_sequence(seq) for seq in y_val]
scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)
rouge_scores = []
for pred, val in zip(decoded_preds, decoded_vals):
    rouge_scores.append(scorer.score(val, pred))
# Aggregate ROUGE scores
avg_rouge1 = np.mean([score['rouge1'].fmeasure for score in rouge_scores])
avg_rougeL = np.mean([score['rougeL'].fmeasure for score in rouge_scores])
print(f"Average ROUGE-1: {avg_rouge1}")
print(f"Average ROUGE-L: {avg_rougeL}")


Average ROUGE-1: 0.05402650356778797
Average ROUGE-L: 0.05402650356778797


In [35]:
import nltk
nltk.download('punkt_tab')
nltk.download('punkt')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [40]:
from nltk.translate.bleu_score import sentence_bleu
bleu_scores = []
for pred, val in zip(decoded_preds, decoded_vals):
    pred_tokens = nltk.word_tokenize(pred)
    val_tokens = nltk.word_tokenize(val)
    bleu_scores.append(sentence_bleu([val_tokens], pred_tokens))
avg_bleu = np.mean(bleu_scores)
print(f"Average BLEU score: {avg_bleu}")

Average BLEU score: 9.842721247767902e-233
